# LOGIKA v73: GBDT + repayment Markov chain

Main features plus a per-customer repayment Markov chain (f251), then the 6 gradient boosting models of the v22 family with validation_v2 over seeds 1, 11, 101, 777 and 31337. Submissions are equal-weight rank means.

In [ ]:
%pip install -q polars==2.0.0 lightgbm==4.7.0 xgboost==3.4.1 catboost==1.2.10

In [ ]:
import os, pathlib
CODE = pathlib.Path('/kaggle/working/code/eda')
CODE.mkdir(parents=True, exist_ok=True)
FILES = {"common.py": "from pathlib import Path\nimport polars as pl\n\nINPUT = next(p.parent for p in Path('/kaggle/input').rglob('train_applications.csv'))\n\nROOT = Path(__file__).resolve().parent.parent\nDATA = Path(INPUT)\nPQ = ROOT / \"data_pq\"\nOUT = ROOT / \"eda\" / \"out\"\nOUT.mkdir(parents=True, exist_ok=True)\nPQ.mkdir(exist_ok=True)\n\nTABLES = [\"train_applications\", \"test_applications\", \"customer_profiles\", \"previous_applications\",\n          \"repayment_history\", \"account_behavior\", \"bureau_observations\", \"ecommerce_transactions\"]\n\n\ndef load(name: str) -> pl.DataFrame:\n    p = PQ / f\"{name}.parquet\"\n    if not p.exists():\n        pl.read_csv(DATA / f\"{name}.csv\", try_parse_dates=True, infer_schema_length=100000).write_parquet(p)\n    return pl.read_parquet(p)\n\n\ndef ks(y, s):\n    \"\"\"KS exactly as the competition scorer (ks_2samp on bad vs good).\"\"\"\n    import numpy as np\n    from scipy.stats import ks_2samp\n    y, s = np.asarray(y), np.asarray(s, dtype=float)\n    m = ~np.isnan(s)\n    y, s = y[m], s[m]\n    if y.min() == y.max():\n        return 0.0\n    return float(ks_2samp(s[y == 1], s[y == 0]).statistic)\n\n\ndef ks_nan(y, s):\n    \"\"\"Univariate KS where NaN is its own bin (placed at either extreme, best taken).\"\"\"\n    import numpy as np\n    s = np.asarray(s, dtype=float)\n    if np.isnan(s).all():\n        return 0.0\n    lo, hi = np.nanmin(s) - 1, np.nanmax(s) + 1\n    return max(ks(y, np.where(np.isnan(s), lo, s)), ks(y, np.where(np.isnan(s), hi, s)))\n\n\ndef psi(a, b, bins=10):\n    \"\"\"Population stability index of b vs a using a's quantile bins (NaN = own bin).\"\"\"\n    import numpy as np\n    a, b = np.asarray(a, float), np.asarray(b, float)\n    qs = np.unique(np.nanquantile(a, np.linspace(0, 1, bins + 1))) if (~np.isnan(a)).any() else np.array([0.0])\n    def dist(x):\n        idx = np.where(np.isnan(x), -1, np.clip(np.searchsorted(qs[1:-1], x, side=\"right\"), 0, None))\n        cnt = np.array([(idx == k).sum() for k in range(-1, len(qs))], float) + 0.5\n        return cnt / cnt.sum()\n    pa, pb = dist(a), dist(b)\n    return float(((pb - pa) * np.log(pb / pa)).sum())\n\n\ndef features():\n    p = PQ / \"features.parquet\"\n    if not p.exists():\n        from features import build_all\n        build_all({t: load(t) for t in TABLES}).write_parquet(p)\n    return pl.read_parquet(p)\n\n\ndef analyze(df: pl.DataFrame, cols, tag: str, top_plot: int = 12):\n    \"\"\"Prints KS / PSI / coverage table and saves decile bad-rate plots for the top features.\"\"\"\n    import numpy as np\n    import matplotlib\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n    tr, te = df.filter(pl.col(\"is_train\") == 1), df.filter(pl.col(\"is_train\") == 0)\n    y = tr[\"target\"].to_numpy()\n    rows = []\n    for c in cols:\n        if tr[c].dtype == pl.String:\n            rate = tr.group_by(c).agg(pl.col(\"target\").mean()).rename({\"target\": \"r\"})\n            s_tr = tr.join(rate, on=c, how=\"left\")[\"r\"].to_numpy()\n            s_te = te.join(rate, on=c, how=\"left\")[\"r\"].to_numpy()\n        else:\n            s_tr, s_te = tr[c].cast(pl.Float64).to_numpy(), te[c].cast(pl.Float64).to_numpy()\n        rows.append((c, ks_nan(y, s_tr), psi(s_tr, s_te), float(np.mean(~np.isnan(s_tr))),\n                     float(np.mean(~np.isnan(s_te))), ks(np.r_[np.zeros(len(s_tr)), np.ones(len(s_te))],\n                                                          np.r_[np.nan_to_num(s_tr, nan=-1e18), np.nan_to_num(s_te, nan=-1e18)])))\n    res = pl.DataFrame(rows, schema=[\"feature\", \"ks_target\", \"psi_test\", \"cov_train\", \"cov_test\", \"ks_train_vs_test\"],\n                       orient=\"row\").sort(\"ks_target\", descending=True)\n    with pl.Config(tbl_rows=200, fmt_str_lengths=40, float_precision=4):\n        print(f\"\\n=== {tag}: univariate KS vs target, drift train->test\")\n        print(res)\n    res.write_csv(OUT / f\"{tag}_ks.csv\")\n    top = res[\"feature\"].head(top_plot).to_list()\n    n = len(top); ncol = 4; nrow = (n + ncol - 1) // ncol\n    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.2 * nrow), squeeze=False)\n    for ax, c in zip(axes.ravel(), top):\n        s = tr.select(c, \"target\")\n        if s[c].dtype == pl.String:\n            g = s.group_by(c).agg(r=pl.col(\"target\").mean(), n=pl.len()).sort(c)\n            ax.bar(g[c].cast(str), g[\"r\"]); ax.tick_params(axis=\"x\", rotation=45)\n        else:\n            s = s.with_columns(b=pl.when(pl.col(c).is_null() | pl.col(c).is_nan()).then(-1)\n                               .otherwise(pl.col(c).rank(\"ordinal\") * 10 // (pl.col(c).is_not_nan().sum() + 1)))\n            g = s.group_by(\"b\").agg(r=pl.col(\"target\").mean(), n=pl.len()).sort(\"b\")\n            ax.bar(g[\"b\"].cast(str).str.replace(\"-1\", \"NA\"), g[\"r\"], color=[\"grey\" if v == -1 else \"C0\" for v in g[\"b\"]])\n        ax.axhline(y.mean(), color=\"r\", ls=\"--\", lw=1)\n        ax.set_title(f\"{c}\\nKS={res.filter(pl.col('feature') == c)['ks_target'][0]:.3f}\", fontsize=9)\n    for ax in axes.ravel()[n:]:\n        ax.axis(\"off\")\n    fig.suptitle(f\"{tag}: bad rate by decile (NA=grey, red=base rate)\")\n    plt.tight_layout(); plt.savefig(OUT / f\"{tag}_deciles.png\", dpi=80); plt.close()\n    return res\n\n\n# ---- residual analysis against the v1 model (results/v1 OOF; no model is trained here) ----\ndef v1_oof():\n    \"\"\"Rank-average of v1 GBDT OOF predictions, aligned to the train order of features().\"\"\"\n    import numpy as np\n    from scipy.stats import rankdata\n    s = np.mean([rankdata(np.load(ROOT / \"results\" / \"v1\" / f\"oof_{n}.npy\")) for n in (\"lgb\", \"xgb\", \"cat\")], 0) / 70000\n    # align by application_id: the row order of a rebuilt feature matrix is not guaranteed to match the v1 run\n    v1 = pl.read_parquet(PQ / \"v1_train_order.parquet\").with_columns(s=pl.Series(s))\n    cur = features().filter(pl.col(\"is_train\") == 1).select(\"application_id\")\n    return cur.join(v1, on=\"application_id\", how=\"left\", maintain_order=\"left\")[\"s\"].to_numpy()\n\n\ndef calibrate(y, s, bins=50):\n    \"\"\"Score -> probability via bad rate in quantile bins of s (monotone enough for residual tests).\"\"\"\n    import numpy as np\n    b = np.minimum((np.argsort(np.argsort(s)) * bins) // len(s), bins - 1)\n    rate = np.bincount(b, weights=y, minlength=bins) / np.bincount(b, minlength=bins)\n    return rate[b]\n\n\ndef qbin(x, bins=10):\n    \"\"\"Quantile bin id (0..bins-1); NaN -> bins (own bin). Ties stay together.\"\"\"\n    import numpy as np\n    x = np.asarray(x, float)\n    out = np.full(len(x), bins)\n    m = ~np.isnan(x)\n    if m.sum() == 0:\n        return out\n    qs = np.unique(np.quantile(x[m], np.linspace(0, 1, bins + 1))[1:-1])\n    out[m] = np.searchsorted(qs, x[m], side=\"right\")\n    return out\n\n\ndef resid_test(y, p, x, bins=10):\n    \"\"\"Does x explain what p (calibrated v1 OOF) missed? Returns (chi2/df, max |O/E-1| over bins >=500 rows, cross-fit dKS).\n    chi2 uses binomial variance per bin; dKS = KS gain of logit(p)+log(O/E)_bin with bin effects fit on the other half.\"\"\"\n    import numpy as np\n    b = qbin(x, bins)\n    o = np.bincount(b, weights=y, minlength=bins + 1)\n    e = np.bincount(b, weights=p, minlength=bins + 1)\n    v = np.bincount(b, weights=p * (1 - p), minlength=bins + 1)\n    n = np.bincount(b, minlength=bins + 1)\n    k = n > 0\n    chi = float((((o - e) ** 2)[k] / np.maximum(v[k], 1e-9)).sum() / max(k.sum() - 1, 1))\n    big = n >= 500\n    lift = float(np.abs(o[big] / np.maximum(e[big], 1e-9) - 1).max()) if big.any() else 0.0\n    rng = np.random.default_rng(0)\n    half = rng.random(len(y)) < 0.5\n    lp = np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4)))\n    adj = np.zeros(len(y))\n    for fit in (half, ~half):\n        of = np.bincount(b[fit], weights=y[fit], minlength=bins + 1) + 5\n        ef = np.bincount(b[fit], weights=p[fit], minlength=bins + 1) + 5\n        adj[~fit] = np.log(of / ef)[b[~fit]]\n    return chi, lift, ks(y, lp + adj) - ks(y, lp)\n\n\ndef screen(cands: pl.DataFrame, tag: str, show: int = 40):\n    \"\"\"Screen candidate features (customer_id + columns) against v1: univariate KS, residual chi2/dKS vs v1 OOF,\n    PSI train->test, and max |Spearman| with any v1 feature (redundancy). Writes out/{tag}_screen.csv.\"\"\"\n    import numpy as np\n    f = features()\n    tr = f.filter(pl.col(\"is_train\") == 1)\n    te = f.filter(pl.col(\"is_train\") == 0)\n    y = tr[\"target\"].to_numpy()\n    p = calibrate(y, v1_oof())\n    ctr = tr.select(\"customer_id\").join(cands, on=\"customer_id\", how=\"left\", maintain_order=\"left\")\n    cte = te.select(\"customer_id\").join(cands, on=\"customer_id\", how=\"left\", maintain_order=\"left\")\n    ID = {\"application_id\", \"customer_id\", \"decision_datetime\", \"target\", \"is_train\"}\n    v1 = [c for c in f.columns if c not in ID and f[c].dtype != pl.String]\n    sub = np.random.default_rng(0).choice(len(y), 15000, replace=False)\n    V = tr[sub].select(v1).with_columns(pl.all().cast(pl.Float64).rank()).to_numpy()\n    rows = []\n    for c in cands.columns:\n        if c == \"customer_id\":\n            continue\n        x = ctr[c].cast(pl.Float64).fill_nan(None).to_numpy().astype(float)\n        xt = cte[c].cast(pl.Float64).fill_nan(None).to_numpy().astype(float)\n        chi, lift, dks = resid_test(y, p, x)\n        xs = pl.Series(x[sub]).rank().to_numpy()\n        m = ~np.isnan(xs)\n        red = 0.0\n        if m.sum() > 100:\n            Vm = V[m]; xm = xs[m]\n            ok = ~np.isnan(Vm)\n            # pairwise-complete spearman via ranks (approx: drop rows with NaN per column)\n            cors = []\n            for j in range(V.shape[1]):\n                k = ok[:, j]\n                if k.sum() > 100 and np.std(Vm[k, j]) > 0 and np.std(xm[k]) > 0:\n                    cors.append(abs(np.corrcoef(xm[k], Vm[k, j])[0, 1]))\n            red = max(cors) if cors else 0.0\n        rows.append((c, ks_nan(y, x), chi, dks, float(np.mean(~np.isnan(x))), psi(x, xt), red))\n    r = pl.DataFrame(rows, schema=[\"feature\", \"ks\", \"chi2_df\", \"dKS_cf\", \"cov\", \"psi\", \"max_rho_v1\"], orient=\"row\")\n    r = r.sort(\"chi2_df\", descending=True)\n    with pl.Config(tbl_rows=show, float_precision=4, fmt_str_lengths=45):\n        print(f\"\\n=== {tag}: candidates vs v1 residual (noise: chi2/df p95~1.2, dKS p95~+0.002)\")\n        print(r.head(show))\n    r.write_csv(OUT / f\"{tag}_screen.csv\")\n    return r\n\n\ndef v2_oof():\n    \"\"\"Equal rank-blend of the six v2 GBDT OOF predictions (results/v2). The v2 run used the same build_all code\n    as the local features(), and the row order was verified (lgb OOF KS 0.5118 matches the notebook).\"\"\"\n    import numpy as np\n    from scipy.stats import rankdata\n    R = ROOT / \"results\" / \"v2\"\n    return np.mean([rankdata(np.load(R / f\"oof_{n}.npy\")) for n in (\"lgb\", \"lgb_s\", \"xgb\", \"xgb_s\", \"cat\", \"cat_s\")], 0) / 70000\n\n\ndef v5_oof():\n    \"\"\"The submitted v5 blend (weights [4,0,0,1,9,3,0,13]/30 reconstructed from the run) as OOF ranks in features() order.\"\"\"\n    import numpy as np\n    from scipy.stats import rankdata\n    R = ROOT / \"results\" / \"v5\"\n    w = {\"lgb\": 4, \"xgb_s\": 1, \"cat\": 9, \"cat_s\": 3, \"tabm\": 13}\n    return sum(v * rankdata(np.load(R / f\"oof_{n}.npy\")) for n, v in w.items()) / 30 / 70000\n\n\ndef v6_oof():\n    \"\"\"The submitted v6 blend (greedy weights [3,1,0,0,2,1,6,10,7]/30) as OOF ranks in features() order.\"\"\"\n    import numpy as np\n    from scipy.stats import rankdata\n    R = ROOT / \"results\" / \"v6\"\n    w = {\"lgb\": 3, \"lgb_s\": 1, \"cat\": 2, \"cat_s\": 1, \"tabpfn\": 6, \"tabm\": 10, \"tabm_p\": 7}\n    return sum(v * rankdata(np.load(R / f\"oof_{n}.npy\")) for n, v in w.items()) / 30 / 70000\n", "features.py": "\"\"\"Leak-safe, customer-level feature builders (one selected application per customer).\n\nAll auxiliary tables are already truncated at decision_datetime (verified in 01_keys_target_time.py),\nbut every builder still filters on the cutoff so the code stays correct if that ever changes.\n\"\"\"\nimport polars as pl\n\nDAY = 86400.0\n\n\ndef _days(a, b):\n    return (pl.col(a) - pl.col(b)).dt.total_seconds() / DAY\n\n\ndef app_features(apps: pl.DataFrame, prof: pl.DataFrame) -> pl.DataFrame:\n    d = apps.join(prof, on=\"customer_id\", how=\"left\")\n    return d.select(\n        \"customer_id\",\n        \"loan_amount\", \"annual_interest_rate\", \"proposed_monthly_installment\", \"loan_purpose\",\n        \"age\", \"region\", \"employment_type\", \"employment_tenure_months\", \"declared_monthly_income\",\n        \"dependents\", \"housing_status\",\n        decision_lag_h=(pl.col(\"decision_datetime\") - pl.col(\"application_datetime\")).dt.total_seconds() / 3600,\n        app_hour=pl.col(\"application_datetime\").dt.hour(),\n        app_dow=pl.col(\"application_datetime\").dt.weekday(),\n        dti_declared=pl.col(\"proposed_monthly_installment\") / pl.col(\"declared_monthly_income\"),\n        lti_declared=pl.col(\"loan_amount\") / pl.col(\"declared_monthly_income\"),\n        tenure_over_age=pl.col(\"employment_tenure_months\") / (pl.col(\"age\") * 12),\n        income_per_head=pl.col(\"declared_monthly_income\") / (1 + pl.col(\"dependents\")),\n    )\n\n\ndef bureau_features(apps: pl.DataFrame, bureau: pl.DataFrame) -> pl.DataFrame:\n    cut = apps.select(\"customer_id\", sel_app=pl.col(\"application_id\"), cutoff=pl.col(\"decision_datetime\"))\n    b = bureau.join(cut, on=\"customer_id\").filter(pl.col(\"available_datetime\") <= pl.col(\"cutoff\"))\n    b = b.with_columns(\n        stale_d=_days(\"cutoff\", \"report_datetime\"),\n        delay_d=_days(\"available_datetime\", \"report_datetime\"),\n        is_sel=pl.col(\"application_id\") == pl.col(\"sel_app\"),\n        prov=pl.col(\"bureau_provider\").str.replace(\"Bureau_\", \"\").str.to_lowercase(),\n    )\n    cur = b.filter(\"is_sel\")\n    num = [\"score\", \"active_accounts\", \"outstanding_balance\", \"revolving_credit_limit\",\n           \"revolving_outstanding_balance\", \"utilization\", \"inquiries_last_6m\", \"delinquent_accounts_last_12m\"]\n    agg = cur.group_by(\"customer_id\").agg(\n        bc_n_reports=pl.len(),\n        bc_n_scored=pl.col(\"score\").is_not_null().sum(),\n        bc_score_mean=pl.col(\"score\").mean(), bc_score_min=pl.col(\"score\").min(),\n        bc_score_max=pl.col(\"score\").max(), bc_score_std=pl.col(\"score\").std(),\n        bc_active_max=pl.col(\"active_accounts\").max(),\n        bc_outstanding_max=pl.col(\"outstanding_balance\").max(),\n        bc_revlimit_max=pl.col(\"revolving_credit_limit\").max(),\n        bc_revbal_max=pl.col(\"revolving_outstanding_balance\").max(),\n        bc_util_max=pl.col(\"utilization\").max(), bc_util_mean=pl.col(\"utilization\").mean(),\n        bc_inq_max=pl.col(\"inquiries_last_6m\").max(), bc_inq_mean=pl.col(\"inquiries_last_6m\").mean(),\n        bc_delinq_max=pl.col(\"delinquent_accounts_last_12m\").max(),\n        bc_delinq_sum=pl.col(\"delinquent_accounts_last_12m\").sum(),\n        bc_stale_mean=pl.col(\"stale_d\").mean(), bc_stale_max=pl.col(\"stale_d\").max(),\n        bc_delay_mean=pl.col(\"delay_d\").mean(),\n    )\n    piv = cur.pivot(on=\"prov\", index=\"customer_id\", values=[\"score\", \"utilization\", \"inquiries_last_6m\",\n                                                            \"delinquent_accounts_last_12m\", \"outstanding_balance\"],\n                    aggregate_function=\"first\")\n    piv = piv.rename({c: \"bp_\" + c.replace(\"inquiries_last_6m\", \"inq\").replace(\"delinquent_accounts_last_12m\", \"delinq\")\n                      .replace(\"outstanding_balance\", \"out\").replace(\"utilization\", \"util\")\n                      for c in piv.columns if c != \"customer_id\"})\n    # history: bureau pulled at earlier (previous) applications\n    hist = b.filter(~pl.col(\"is_sel\")).group_by(\"customer_id\").agg(\n        bh_n_pulls=pl.col(\"application_id\").n_unique(),\n        bh_score_mean=pl.col(\"score\").mean(), bh_score_min=pl.col(\"score\").min(),\n        bh_delinq_max=pl.col(\"delinquent_accounts_last_12m\").max(),\n        bh_inq_mean=pl.col(\"inquiries_last_6m\").mean(),\n        bh_last_score=pl.col(\"score\").sort_by(\"report_datetime\", \"bureau_observation_id\").drop_nulls().last(),\n        bh_last_outstanding=pl.col(\"outstanding_balance\").sort_by(\"report_datetime\", \"bureau_observation_id\").last(),\n        bh_pulls_180d=pl.col(\"application_id\").filter(pl.col(\"stale_d\") <= 180).n_unique(),\n    )\n    out = agg.join(piv, on=\"customer_id\", how=\"left\").join(hist, on=\"customer_id\", how=\"left\")\n    return out.with_columns(\n        bh_score_delta=pl.col(\"bc_score_mean\") - pl.col(\"bh_last_score\"),\n        bh_outstanding_delta=pl.col(\"bc_outstanding_max\") - pl.col(\"bh_last_outstanding\"),\n    )\n\n\ndef repayment_features(apps: pl.DataFrame, rep: pl.DataFrame, prev: pl.DataFrame) -> pl.DataFrame:\n    cut = apps.select(\"customer_id\", cutoff=pl.col(\"decision_datetime\"))\n    r = rep.join(cut, on=\"customer_id\").filter(pl.col(\"due_date\") <= pl.col(\"cutoff\"))\n    r = r.with_columns(\n        age_d=_days(\"cutoff\", \"due_date\"),\n        unpaid=pl.col(\"amount_due\") - pl.col(\"amount_paid\"),\n        late_d=pl.when(pl.col(\"amount_paid\") >= pl.col(\"amount_due\")).then(_days(\"last_payment_datetime\", \"due_date\")),\n    )\n    w = lambda days: pl.col(\"age_d\") <= days  # noqa: E731\n    agg = r.group_by(\"customer_id\").agg(\n        rp_n_loans=pl.col(\"loan_id\").n_unique(), rp_n_inst=pl.len(),\n        rp_maxdpd_max=pl.col(\"max_days_past_due\").max(), rp_maxdpd_mean=pl.col(\"max_days_past_due\").mean(),\n        rp_n_dpd1=(pl.col(\"max_days_past_due\") > 0).sum(), rp_n_dpd30=(pl.col(\"max_days_past_due\") >= 30).sum(),\n        rp_n_dpd60=(pl.col(\"max_days_past_due\") >= 60).sum(), rp_n_dpd90=(pl.col(\"max_days_past_due\") >= 90).sum(),\n        rp_frac_dpd1=(pl.col(\"max_days_past_due\") > 0).mean(), rp_frac_dpd30=(pl.col(\"max_days_past_due\") >= 30).mean(),\n        rp_cur_dpd_max=pl.col(\"days_past_due\").max(), rp_n_unpaid=(pl.col(\"unpaid\") > 0).sum(),\n        rp_unpaid_amt=pl.col(\"unpaid\").sum(),\n        rp_paid_ratio=pl.col(\"amount_paid\").sum() / pl.col(\"amount_due\").sum(),\n        rp_late_mean=pl.col(\"late_d\").mean(), rp_late_max=pl.col(\"late_d\").max(),\n        rp_maxdpd_6m=pl.col(\"max_days_past_due\").filter(w(183)).max(),\n        rp_maxdpd_12m=pl.col(\"max_days_past_due\").filter(w(365)).max(),\n        rp_frac_dpd1_6m=(pl.col(\"max_days_past_due\") > 0).filter(w(183)).mean(),\n        rp_late_mean_6m=pl.col(\"late_d\").filter(w(183)).mean(),\n        rp_maxdpd_last3=pl.col(\"max_days_past_due\").sort_by(\"due_date\", \"loan_id\").tail(3).max(),\n        rp_days_since_delinq=pl.col(\"age_d\").filter(pl.col(\"max_days_past_due\") > 0).min(),\n        rp_days_since_due=pl.col(\"age_d\").min(),\n    )\n    # active loans: fewer installments observed than contracted term -> remaining obligation\n    loans = r.group_by(\"loan_id\").agg(customer_id=pl.col(\"customer_id\").first(), n_seen=pl.len(),\n                                      inst=pl.col(\"amount_due\").first())\n    loans = loans.join(prev.select(\"loan_id\", \"term_months\"), on=\"loan_id\", how=\"left\").with_columns(\n        remaining=(pl.col(\"term_months\") - pl.col(\"n_seen\")).clip(lower_bound=0))\n    act = loans.group_by(\"customer_id\").agg(\n        rp_n_active=(pl.col(\"remaining\") > 0).sum(),\n        rp_active_inst=pl.col(\"inst\").filter(pl.col(\"remaining\") > 0).sum(),\n        rp_remaining_oblig=(pl.col(\"remaining\") * pl.col(\"inst\")).sum(),\n    )\n    return agg.join(act, on=\"customer_id\", how=\"left\")\n\n\ndef account_features(apps: pl.DataFrame, acc: pl.DataFrame) -> pl.DataFrame:\n    cut = apps.select(\"customer_id\", cutoff=pl.col(\"decision_datetime\"))\n    a = acc.join(cut, on=\"customer_id\").filter(\n        pl.col(\"month\").dt.offset_by(\"1mo\").cast(pl.Datetime) <= pl.col(\"cutoff\")).sort(\"customer_id\", \"month\")\n    a = a.with_columns(mago=pl.col(\"month\").rank(\"dense\", descending=True).over(\"customer_id\"))\n    l3 = pl.col(\"mago\") <= 3\n    out = a.group_by(\"customer_id\").agg(\n        ac_n_months=pl.len(),\n        ac_inflow_mean=pl.col(\"inflows\").mean(), ac_outflow_mean=pl.col(\"outflows\").mean(),\n        ac_income_mean=pl.col(\"income_like_credits\").mean(), ac_income_std=pl.col(\"income_like_credits\").std(),\n        ac_income_min=pl.col(\"income_like_credits\").min(),\n        ac_avgbal_mean=pl.col(\"average_balance\").mean(), ac_minbal_min=pl.col(\"minimum_balance\").min(),\n        ac_minbal_mean=pl.col(\"minimum_balance\").mean(),\n        ac_lowdays_mean=pl.col(\"low_balance_days\").mean(), ac_lowdays_max=pl.col(\"low_balance_days\").max(),\n        ac_frac_zero_min=(pl.col(\"minimum_balance\") == 0).mean(),\n        ac_net_flow_mean=(pl.col(\"inflows\") - pl.col(\"outflows\")).mean(),\n        ac_out_in_ratio=pl.col(\"outflows\").sum() / pl.col(\"inflows\").sum(),\n        ac_income_l3=pl.col(\"income_like_credits\").filter(l3).mean(),\n        ac_avgbal_l3=pl.col(\"average_balance\").filter(l3).mean(),\n        ac_lowdays_l3=pl.col(\"low_balance_days\").filter(l3).mean(),\n        ac_out_in_l3=pl.col(\"outflows\").filter(l3).sum() / pl.col(\"inflows\").filter(l3).sum(),\n        ac_last_close=pl.col(\"closing_balance\").last(),\n        ac_out_in_b=pl.col(\"outflows\").filter(pl.col(\"mago\").is_between(7, 12)).sum()\n        / pl.col(\"inflows\").filter(pl.col(\"mago\").is_between(7, 12)).sum(),\n        ac_lowdays_b=pl.col(\"low_balance_days\").filter(pl.col(\"mago\").is_between(7, 12)).mean(),\n        ac_avgbal_b=pl.col(\"average_balance\").filter(pl.col(\"mago\").is_between(7, 12)).mean(),\n        ac_out_in_l1=(pl.col(\"outflows\") / pl.col(\"inflows\")).filter(pl.col(\"mago\") == 1).first(),\n        ac_close_slope=pl.cov(pl.col(\"mago\").cast(pl.Float64), pl.col(\"closing_balance\").cast(pl.Float64))\n        / pl.col(\"mago\").cast(pl.Float64).var() * -1,\n    )\n    return out.with_columns(\n        ac_income_cv=pl.col(\"ac_income_std\") / pl.col(\"ac_income_mean\"),\n        ac_income_trend=pl.col(\"ac_income_l3\") / pl.col(\"ac_income_mean\"),\n        ac_bal_trend=pl.col(\"ac_avgbal_l3\") / (pl.col(\"ac_avgbal_mean\") + 1),\n        ac_out_in_delta=pl.col(\"ac_out_in_l3\") - pl.col(\"ac_out_in_b\"),\n        ac_lowdays_delta=pl.col(\"ac_lowdays_l3\") - pl.col(\"ac_lowdays_b\"),\n        ac_bal_delta=(pl.col(\"ac_avgbal_l3\") + 1) / (pl.col(\"ac_avgbal_b\") + 1),\n    )\n\n\ndef ecommerce_features(apps: pl.DataFrame, ec: pl.DataFrame) -> pl.DataFrame:\n    cut = apps.select(\"customer_id\", cutoff=pl.col(\"decision_datetime\"))\n    e = ec.join(cut, on=\"customer_id\").filter(pl.col(\"transaction_datetime\") <= pl.col(\"cutoff\"))\n    e = e.with_columns(age_d=_days(\"cutoff\", \"transaction_datetime\"), done=pl.col(\"order_status\") == \"completed\")\n    # order level: order_total repeats on each item -> dedupe before aggregating\n    o = e.group_by(\"order_id\").agg(pl.col(\"customer_id\", \"age_d\", \"order_total\", \"payment_method\", \"done\").first())\n    agg = o.group_by(\"customer_id\").agg(\n        ec_n_orders=pl.len(), ec_cancel_rate=1 - pl.col(\"done\").mean(),\n        ec_spend=pl.col(\"order_total\").filter(\"done\").sum(),\n        ec_aov=pl.col(\"order_total\").filter(\"done\").mean(),\n        ec_recency_d=pl.col(\"age_d\").min(), ec_tenure_d=pl.col(\"age_d\").max(),\n        ec_orders_7d=(pl.col(\"age_d\") <= 7).sum(), ec_orders_14d=(pl.col(\"age_d\") <= 14).sum(),\n        ec_orders_30d=(pl.col(\"age_d\") <= 30).sum(), ec_orders_60d=(pl.col(\"age_d\") <= 60).sum(),\n        ec_orders_90d=(pl.col(\"age_d\") <= 90).sum(),\n        ec_base_wk=((pl.col(\"age_d\") > 120) & (pl.col(\"age_d\") <= 365)).sum()\n        / ((pl.col(\"age_d\").max().clip(upper_bound=365) - 120).clip(lower_bound=7) / 7),\n        ec_spend_30d=pl.col(\"order_total\").filter(pl.col(\"done\") & (pl.col(\"age_d\") <= 30)).sum(),\n        ec_spend_90d=pl.col(\"order_total\").filter(pl.col(\"done\") & (pl.col(\"age_d\") <= 90)).sum(),\n        ec_cancel_90d=(~pl.col(\"done\")).filter(pl.col(\"age_d\") <= 90).sum(),\n        **{f\"ec_pm_{m}\": (pl.col(\"payment_method\") == m).mean()\n           for m in [\"COD\", \"e_wallet\", \"bank_transfer\", \"debit_card\", \"credit_card\"]},\n    )\n    cat = (e.filter(\"done\").group_by(\"customer_id\", \"product_category\").agg(s=pl.col(\"item_total\").sum())\n           .with_columns(s=pl.col(\"s\") / pl.col(\"s\").sum().over(\"customer_id\"))\n           .pivot(on=\"product_category\", index=\"customer_id\", values=\"s\"))\n    cat = cat.rename({c: f\"ec_cat_{c}\" for c in cat.columns if c != \"customer_id\"})\n    out = agg.join(cat, on=\"customer_id\", how=\"left\").with_columns(\n        ec_spend_per_month=pl.col(\"ec_spend\") / (pl.col(\"ec_tenure_d\") / 30.4 + 1),\n        ec_spend_90d_ratio=pl.col(\"ec_spend_90d\") / 3 / (pl.col(\"ec_spend\") / (pl.col(\"ec_tenure_d\") / 30.4 + 1) + 1),\n    )\n    return out.with_columns(\n        pl.col(\"^ec_cat_.*$\").fill_null(0),\n        ec_accel_30d=(pl.col(\"ec_orders_30d\") / (30 / 7)) / (pl.col(\"ec_base_wk\") + 0.1),\n        ec_accel_90d=(pl.col(\"ec_orders_90d\") / (90 / 7)) / (pl.col(\"ec_base_wk\") + 0.1),\n    )\n\n\ndef prev_features(apps: pl.DataFrame, prev: pl.DataFrame) -> pl.DataFrame:\n    cut = apps.select(\"customer_id\", cutoff=pl.col(\"decision_datetime\"), cur_rate=pl.col(\"annual_interest_rate\"),\n                      cur_amt=pl.col(\"loan_amount\"))\n    p = prev.join(cut, on=\"customer_id\").filter(pl.col(\"decision_datetime\") <= pl.col(\"cutoff\"))\n    p = p.with_columns(age_d=_days(\"cutoff\", \"application_datetime\"))\n    out = p.group_by(\"customer_id\").agg(\n        pv_n=pl.len(), pv_n_appr=(pl.col(\"decision\") == \"approved\").sum(),\n        pv_n_rej=(pl.col(\"decision\") == \"rejected\").sum(), pv_n_canc=(pl.col(\"decision\") == \"cancelled\").sum(),\n        pv_rej_rate=(pl.col(\"decision\") == \"rejected\").mean(),\n        pv_days_since_last=pl.col(\"age_d\").min(), pv_days_since_first=pl.col(\"age_d\").max(),\n        pv_n_90d=(pl.col(\"age_d\") <= 90).sum(), pv_n_180d=(pl.col(\"age_d\") <= 180).sum(),\n        pv_rej_365d=((pl.col(\"decision\") == \"rejected\") & (pl.col(\"age_d\") <= 365)).sum(),\n        pv_rate_mean=pl.col(\"annual_interest_rate\").mean(), pv_rate_max=pl.col(\"annual_interest_rate\").max(),\n        pv_rate_last=pl.col(\"annual_interest_rate\").sort_by(\"application_datetime\", \"application_id\").drop_nulls().last(),\n        pv_appr_ratio=(pl.col(\"approved_amount\") / pl.col(\"requested_amount\")).mean(),\n        pv_req_mean=pl.col(\"requested_amount\").mean(), pv_appr_sum=pl.col(\"approved_amount\").sum(),\n        pv_appr_max=pl.col(\"approved_amount\").max(),\n        pv_last_decision_rej=(pl.col(\"decision\").sort_by(\"application_datetime\", \"application_id\").last() == \"rejected\"),\n        cur_rate=pl.col(\"cur_rate\").first(), cur_amt=pl.col(\"cur_amt\").first(),\n    )\n    return out.with_columns(\n        pv_rate_delta=pl.col(\"cur_rate\") - pl.col(\"pv_rate_last\"),\n        pv_amt_vs_max=pl.col(\"cur_amt\") / pl.col(\"pv_appr_max\"),\n        pv_last_decision_rej=pl.col(\"pv_last_decision_rej\").cast(pl.Int8),\n    ).drop(\"cur_rate\", \"cur_amt\")\n\n\ndef v2_features(apps: pl.DataFrame, tables: dict) -> pl.DataFrame:\n    \"\"\"v2 additions (eda/21-26): signals left in the v1 residual. Same cutoff rules as the v1 builders.\"\"\"\n    cut = apps.select(\"customer_id\", \"application_id\", cutoff=pl.col(\"decision_datetime\"),\n                      inst_new=pl.col(\"proposed_monthly_installment\"), amt_new=pl.col(\"loan_amount\"))\n    # bureau: score trend/range over every pull, newest report, delinquency over all pulls, external inquiries\n    b = tables[\"bureau_observations\"].join(cut.select(\"customer_id\", \"cutoff\", sel=pl.col(\"application_id\")), on=\"customer_id\")\n    b = b.filter(pl.col(\"available_datetime\") <= pl.col(\"cutoff\")).with_columns(\n        age_d=_days(\"cutoff\", \"report_datetime\"), is_sel=pl.col(\"application_id\") == pl.col(\"sel\"))\n    bz = b.group_by(\"customer_id\").agg(\n        bz_trend=pl.cov(pl.col(\"age_d\"), pl.col(\"score\")) / pl.col(\"age_d\").var() * -365,\n        bz_range=pl.col(\"score\").max() - pl.col(\"score\").min(),\n        bz_newest=pl.col(\"score\").sort_by(\"report_datetime\", \"bureau_observation_id\").drop_nulls().last(),\n        bz_delinq_all_mean=pl.col(\"delinquent_accounts_last_12m\").mean(),\n        bz_active_trend=pl.col(\"active_accounts\").filter(\"is_sel\").mean()\n        - pl.col(\"active_accounts\").filter(~pl.col(\"is_sel\")).mean(),\n        bz_inq_cur=pl.col(\"inquiries_last_6m\").filter(\"is_sel\").mean(),\n    )\n    p = tables[\"previous_applications\"].join(cut, on=\"customer_id\").filter(pl.col(\"decision_datetime\") <= pl.col(\"cutoff\"))\n    own6 = p.filter(_days(\"cutoff\", \"application_datetime\") <= 183).group_by(\"customer_id\").agg(own6=pl.len())\n    bz = bz.join(own6, on=\"customer_id\", how=\"left\").with_columns(\n        bz_inq_external=pl.col(\"bz_inq_cur\") - pl.col(\"own6\").fill_null(0)).drop(\"bz_inq_cur\", \"own6\")\n    # previous approvals: tenor mix, recency, recent approved volume, remaining scheduled obligation\n    ap = p.filter(pl.col(\"decision\") == \"approved\").with_columns(\n        age_d=_days(\"cutoff\", \"decision_datetime\"),\n        rem=(pl.col(\"term_months\") - (_days(\"cutoff\", \"decision_datetime\") / 30.44).floor()).clip(lower_bound=0))\n    ob = ap.group_by(\"customer_id\").agg(\n        ob_term_mean=pl.col(\"term_months\").mean(), ob_days_since_appr=pl.col(\"age_d\").min(),\n        ob_amt_90d=pl.col(\"approved_amount\").filter(pl.col(\"age_d\") <= 90).sum(),\n        ob_remaining=(pl.col(\"scheduled_monthly_installment\") * pl.col(\"rem\")).sum(),\n        amt_new=pl.col(\"amt_new\").first(),\n    ).with_columns(ob_amt_vs_last=pl.col(\"amt_new\") / (pl.col(\"ob_amt_90d\") + 1e6)).drop(\"amt_new\")\n    # repayment: first installment, latest loan, 1-29 dpd share, recent trend, streaks, severity when late\n    r = tables[\"repayment_history\"].join(cut.select(\"customer_id\", \"cutoff\"), on=\"customer_id\")\n    # loan_id breaks due_date ties between concurrent loans -> deterministic order\n    r = r.filter(pl.col(\"due_date\") <= pl.col(\"cutoff\")).sort(\"customer_id\", \"due_date\", \"loan_id\").with_columns(\n        late=(pl.col(\"max_days_past_due\") > 0).cast(pl.Int32),\n        paid=pl.col(\"amount_paid\") >= pl.col(\"amount_due\"),\n        late_d=_days(\"last_payment_datetime\", \"due_date\"),\n        k=pl.col(\"due_date\").rank(\"ordinal\").over(\"loan_id\"),\n        loan_start=pl.col(\"due_date\").min().over(\"loan_id\"))\n    # streak of consecutive late installments within one loan (concurrent loans interleave in time)\n    r = r.with_columns(run=(pl.col(\"late\") != pl.col(\"late\").shift(1)).fill_null(True).cum_sum().over(\"loan_id\"))\n    streak = (r.filter(pl.col(\"late\") == 1).group_by(\"customer_id\", \"loan_id\", \"run\").agg(n=pl.len())\n              .group_by(\"customer_id\").agg(rq_streak_max=pl.col(\"n\").max()))\n    last = (r.filter(pl.col(\"loan_start\") == pl.col(\"loan_start\").max().over(\"customer_id\"))\n            .group_by(\"customer_id\").agg(rq_lastloan_late=pl.col(\"late\").mean(), rq_lastloan_maxdpd=pl.col(\"max_days_past_due\").max()))\n    dpd, late = pl.col(\"max_days_past_due\"), pl.col(\"max_days_past_due\") > 0\n    rq = r.group_by(\"customer_id\").agg(\n        rq_first_inst_late=pl.col(\"late\").filter(pl.col(\"k\") == 1).mean(),\n        rq_frac_1_29=(late & (dpd < 30)).mean(),\n        rq_late_trend=pl.col(\"late\").tail(6).mean() - pl.col(\"late\").head(pl.len() - 6).mean(),\n        sv_maxdpd_mean_late=dpd.filter(late).mean(),\n        sv_late_d_mean=pl.col(\"late_d\").filter(pl.col(\"paid\") & (pl.col(\"late_d\") > 0)).mean(),\n        sv_share30_of_late=(dpd >= 30).sum() / late.sum(),\n        sv_unpaid_of_late=(~pl.col(\"paid\") & late).sum() / late.sum(),\n    ).join(streak, on=\"customer_id\", how=\"left\").join(last, on=\"customer_id\", how=\"left\")\n    # account: non-income inflows, recent outflows, worst free cash flow, new installment vs free cash / min balance\n    a = tables[\"account_behavior\"].join(cut.select(\"customer_id\", \"cutoff\", \"inst_new\"), on=\"customer_id\")\n    a = a.filter(pl.col(\"month\").dt.offset_by(\"1mo\").cast(pl.Datetime) <= pl.col(\"cutoff\")).with_columns(\n        l3=pl.col(\"month\").rank(\"dense\", descending=True).over(\"customer_id\") <= 3,\n        free=pl.col(\"inflows\") - pl.col(\"outflows\"))\n    cf = a.group_by(\"customer_id\").agg(\n        cf_nonincome_share=1 - pl.col(\"income_like_credits\").sum() / pl.col(\"inflows\").sum(),\n        cf_nonincome_l3=1 - pl.col(\"income_like_credits\").filter(\"l3\").sum() / pl.col(\"inflows\").filter(\"l3\").sum(),\n        cf_out_l3=pl.col(\"outflows\").filter(\"l3\").mean(), cf_free_min=pl.col(\"free\").min(),\n        cf_free_l3=pl.col(\"free\").filter(\"l3\").mean(), cf_inc_l3=pl.col(\"income_like_credits\").filter(\"l3\").mean(),\n        cf_minbal_l3=pl.col(\"minimum_balance\").filter(\"l3\").min(), inst_new=pl.col(\"inst_new\").first(),\n    ).with_columns(\n        cf_inst_to_free=pl.col(\"inst_new\") / (pl.col(\"cf_free_l3\").clip(lower_bound=0) + 1e5),\n        cf_out_to_inc_l3=pl.col(\"cf_out_l3\") / (pl.col(\"cf_inc_l3\") + 1e5),\n        cf_inst_to_minbal=pl.col(\"inst_new\") / (pl.col(\"cf_minbal_l3\") + 1e5),\n    ).drop(\"inst_new\", \"cf_free_l3\", \"cf_inc_l3\")\n    # e-commerce: cancellations, night share, basket size, AOV shift, tightest gap, smooth recency-weighted counts\n    e = tables[\"ecommerce_transactions\"].join(cut.select(\"customer_id\", \"cutoff\"), on=\"customer_id\")\n    e = e.filter(pl.col(\"transaction_datetime\") <= pl.col(\"cutoff\"))\n    o = e.group_by(\"order_id\").agg(pl.col(\"customer_id\", \"cutoff\", \"transaction_datetime\", \"order_total\", \"order_status\").first(),\n                                   n_items=pl.len())\n    o = o.with_columns(age_d=_days(\"cutoff\", \"transaction_datetime\"), done=pl.col(\"order_status\") == \"completed\",\n                       night=pl.col(\"transaction_datetime\").dt.hour() < 5)\n    r28, old = pl.col(\"age_d\") <= 28, pl.col(\"age_d\") > 182\n    ec = o.group_by(\"customer_id\").agg(\n        ec2_base=(old & (pl.col(\"age_d\") <= 730)).sum()\n        / ((pl.col(\"age_d\").max().clip(upper_bound=730) - 182).clip(lower_bound=14) / 7),\n        ec2_cancel_28=(~pl.col(\"done\")).filter(r28).sum(),\n        ec2_night_share_90=pl.col(\"night\").filter(pl.col(\"age_d\") <= 90).mean(),\n        ec2_items_per_order_28=pl.col(\"n_items\").filter(r28).mean(),\n        ec2_aov_28=pl.col(\"order_total\").filter(pl.col(\"done\") & r28).mean(),\n        ec2_aov_old=pl.col(\"order_total\").filter(pl.col(\"done\") & old).mean(),\n        ec2_gap_min_28=pl.col(\"age_d\").filter(r28).sort().diff().min(),\n        **{f\"ec2_ewm{t}\": (-pl.col(\"age_d\") / t).exp().sum() for t in (7, 28, 56)},\n    ).with_columns(\n        ec2_aov_shift=pl.col(\"ec2_aov_28\") / pl.col(\"ec2_aov_old\"),\n        ec2_ewm7_rel=pl.col(\"ec2_ewm7\") / (pl.col(\"ec2_base\") + 0.05),\n        ec2_ewm28_exc=pl.col(\"ec2_ewm28\") - pl.col(\"ec2_base\") * 4,\n        ec2_ewm56_rel=pl.col(\"ec2_ewm56\") / (pl.col(\"ec2_base\") * 8 + 0.05),\n    ).drop(\"ec2_aov_28\", \"ec2_aov_old\", \"ec2_ewm7\", \"ec2_ewm28\", \"ec2_ewm56\")\n    out = cut.select(\"customer_id\")\n    for part in (bz, ob, rq, cf, ec):\n        out = out.join(part, on=\"customer_id\", how=\"left\")\n    return out\n\n\nPP_CATS = [\"groceries\", \"fashion\", \"electronics\", \"home\", \"health\", \"education\", \"personal_care\"]\n\n\ndef v5_features(apps: pl.DataFrame, tables: dict, *, reference_customers=None) -> pl.DataFrame:\n    \"\"\"Price features with an expanding reference before each decision month.\n\n    New inductive runs must supply fitting-partition customer IDs. None preserves\n    the historical transductive recipe for reproducing existing artifacts.\n    Query events remain available only for their own customer aggregates.\n    \"\"\"\n    cut = apps.select(\"customer_id\", cutoff=pl.col(\"decision_datetime\"), dm=pl.col(\"decision_datetime\").dt.truncate(\"1mo\"))\n    raw = tables[\"ecommerce_transactions\"].with_columns(lu=(pl.col(\"item_total\") / pl.col(\"quantity\")).log(),\n                                                        ym=pl.col(\"transaction_datetime\").dt.truncate(\"1mo\"))\n    reference = raw\n    if reference_customers is not None:\n        customers = list(reference_customers)\n        if not customers or len(customers) != len(set(customers)):\n            raise ValueError(\"Reference customer IDs must be nonempty and unique\")\n        if not set(customers) <= set(apps[\"customer_id\"].to_list()):\n            raise ValueError(\"Reference customers must belong to the application frame\")\n        reference = raw.filter(pl.col(\"customer_id\").is_in(customers))\n    ms = reference.group_by(\"product_category\", \"ym\").agg(n=pl.len(), s=pl.col(\"lu\").sum(), ss=(pl.col(\"lu\") ** 2).sum())\n    ref = (cut.select(\"dm\").unique().join(ms, how=\"cross\").filter(pl.col(\"ym\") < pl.col(\"dm\"))\n           .group_by(\"product_category\", \"dm\").agg(pl.col(\"n\", \"s\", \"ss\").sum())\n           .with_columns(mu=pl.col(\"s\") / pl.col(\"n\"),\n                         sd=((pl.col(\"ss\") - pl.col(\"s\") ** 2 / pl.col(\"n\")) / (pl.col(\"n\") - 1)).sqrt()))\n    e = raw.join(cut, on=\"customer_id\").filter(pl.col(\"transaction_datetime\") <= pl.col(\"cutoff\"))\n    e = e.join(ref.select(\"product_category\", \"dm\", \"mu\", \"sd\"), on=[\"product_category\", \"dm\"], how=\"left\")\n    e = e.with_columns(age_d=_days(\"cutoff\", \"transaction_datetime\"), done=pl.col(\"order_status\") == \"completed\",\n                       z=(pl.col(\"lu\") - pl.col(\"mu\")) / pl.col(\"sd\"))\n    lo, hi = pl.col(\"z\").quantile(0.1), pl.col(\"z\").quantile(0.9)\n    pp = e.group_by(\"customer_id\").agg(\n        pp_mean=pl.col(\"z\").mean(),\n        pp_trim=pl.col(\"z\").filter((pl.col(\"z\") > lo) & (pl.col(\"z\") < hi)).mean(),\n        pp_done=pl.col(\"z\").filter(\"done\").mean(), pp_min=pl.col(\"z\").min(), pp_max=pl.col(\"z\").max(),\n        pp_30=pl.col(\"z\").filter(pl.col(\"age_d\") <= 30).mean(), pp_90=pl.col(\"z\").filter(pl.col(\"age_d\") <= 90).mean(),\n        pp_old=pl.col(\"z\").filter(pl.col(\"age_d\") > 182).mean(),\n        pp_qty_item=pl.col(\"quantity\").mean(), pp_qty_item_90=pl.col(\"quantity\").filter(pl.col(\"age_d\") <= 90).mean(),\n        **{f\"pp_cat_{c}\": pl.col(\"z\").filter(pl.col(\"product_category\") == c).mean() for c in PP_CATS},\n    )\n    month = apps.select(\"customer_id\", dec_month=pl.col(\"decision_datetime\").dt.month())\n    return month.join(pp, on=\"customer_id\", how=\"left\")\n\n\ndef build_all(tables: dict) -> pl.DataFrame:\n    \"\"\"tables: dict of polars frames keyed by table name. Returns one row per selected application.\"\"\"\n    tr = tables[\"train_applications\"].with_columns(is_train=pl.lit(1))\n    te = tables[\"test_applications\"].with_columns(target=pl.lit(None, pl.Int64), is_train=pl.lit(0))\n    apps = pl.concat([tr, te], how=\"diagonal_relaxed\")\n    f = apps.select(\"application_id\", \"customer_id\", \"decision_datetime\", \"target\", \"is_train\")\n    for part in [\n        app_features(apps, tables[\"customer_profiles\"]),\n        bureau_features(apps, tables[\"bureau_observations\"]),\n        repayment_features(apps, tables[\"repayment_history\"], tables[\"previous_applications\"]),\n        account_features(apps, tables[\"account_behavior\"]),\n        ecommerce_features(apps, tables[\"ecommerce_transactions\"]),\n        prev_features(apps, tables[\"previous_applications\"]),\n        v2_features(apps, tables),\n        v5_features(apps, tables),\n    ]:\n        f = f.join(part, on=\"customer_id\", how=\"left\")\n    # cross-table ratios: the real affordability picture\n    f = f.with_columns(\n        has_history=pl.col(\"pv_n\").is_not_null().cast(pl.Int8),\n        dti_verified=pl.col(\"proposed_monthly_installment\") / pl.col(\"ac_income_mean\"),\n        dti_total=(pl.col(\"proposed_monthly_installment\") + pl.col(\"rp_active_inst\").fill_null(0))\n        / pl.col(\"declared_monthly_income\"),\n        income_verif_ratio=pl.col(\"ac_income_mean\") / pl.col(\"declared_monthly_income\"),\n        debt_to_income=pl.col(\"bc_outstanding_max\") / pl.col(\"declared_monthly_income\"),\n        inst_to_avgbal=pl.col(\"proposed_monthly_installment\") / (pl.col(\"ac_avgbal_mean\") + 1),\n        ecspend_to_income=pl.col(\"ec_spend_per_month\") / pl.col(\"declared_monthly_income\"),\n    )\n    return f\n", "valid2.py": "\"\"\"validation_v2: one protocol for local screening (one fold) and full Kaggle runs.\n\nOuter: StratifiedKFold(5, shuffle, seed) per seed; the outer validation fold is only scored, never used for any decision.\nInner: a stratified 1/7 split of the outer training fold picks the iteration / epoch (early stopping). The model is then\nrefitted on the whole outer training fold with that fixed budget (GBDT: best x REFIT, TabM: best epoch).\nEvery run writes OOF and test predictions with application_id. Blend weights are chosen later, cross-fitted over the\nouter folds (eda/blend2.py), never on the rows they are scored on.\"\"\"\nimport gc\nimport json\nimport math\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport polars as pl\nfrom scipy.stats import ks_2samp\nfrom sklearn.model_selection import StratifiedKFold\n\nID_COLS = [\"application_id\", \"customer_id\", \"decision_datetime\", \"target\", \"is_train\"]\nCAT_COLS = [\"loan_purpose\", \"region\", \"employment_type\", \"housing_status\"]\nN_FOLDS, INNER, REFIT = 5, 7, 1.15\nSEEDS = [42, 7, 2026]\nN_ROUNDS, EARLY_STOP = 10000, 200\nTHREADS = os.cpu_count() or 4\n\nLGB = dict(objective=\"binary\", metric=\"auc\", learning_rate=0.02, num_leaves=31, min_child_samples=100,\n           feature_fraction=0.5, bagging_fraction=0.8, bagging_freq=1, lambda_l2=5.0, verbose=-1, n_jobs=THREADS)\nLGB_S = dict(LGB, num_leaves=7, max_depth=3, min_child_samples=200, feature_fraction=0.4, lambda_l2=10.0)\nXGB = dict(objective=\"binary:logistic\", eval_metric=\"auc\", learning_rate=0.02, max_depth=5, min_child_weight=5,\n           subsample=0.8, colsample_bytree=0.5, reg_lambda=5.0, tree_method=\"hist\", device=\"cuda\")\nXGB_S = dict(XGB, max_depth=3, min_child_weight=20, colsample_bytree=0.4, reg_lambda=10.0)\nCAT = dict(learning_rate=0.03, depth=6, l2_leaf_reg=5, loss_function=\"Logloss\", eval_metric=\"AUC\",\n           task_type=\"GPU\", verbose=0)\nCAT_S = dict(CAT, depth=4, l2_leaf_reg=10)\nLGB_X = dict(LGB, extra_trees=True)\nXGB_D8 = dict(XGB, max_depth=8, min_child_weight=20, colsample_bytree=0.4, reg_lambda=10.0)\nCAT_D8 = dict(CAT, depth=8, l2_leaf_reg=10)\nTABM = dict(lr=2e-3, wd=3e-4, bins=48, batch=512, epochs=60, patience=8)\n\n\ndef ks(y, p):\n    return float(ks_2samp(p[y == 1], p[y == 0]).statistic)\n\n\ndef outer(y, seed):\n    return list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(y)), y))\n\n\ndef inner(y_tr, seed, fold):\n    \"\"\"(fit, stop) positions inside the outer training fold.\"\"\"\n    return next(StratifiedKFold(INNER, shuffle=True, random_state=1000 * seed + fold).split(np.zeros(len(y_tr)), y_tr))\n\n\ndef frames(features: pl.DataFrame):\n    \"\"\"features (build_all output) -> X, Xt (pandas; categoricals typed with train categories), y, train ids, test ids.\"\"\"\n    f = features.sort(\"application_id\")\n    tr, te = f.filter(pl.col(\"is_train\") == 1), f.filter(pl.col(\"is_train\") == 0)\n    cols = sorted(c for c in f.columns if c not in ID_COLS)\n    X, Xt = tr.select(cols).to_pandas(), te.select(cols).to_pandas()\n    for c in cols:\n        if c in CAT_COLS or f[c].dtype == pl.String:\n            cats = sorted(X[c].dropna().unique())\n            X[c], Xt[c] = pd.Categorical(X[c], categories=cats), pd.Categorical(Xt[c], categories=cats)\n        else:\n            X[c] = X[c].astype(float).replace([np.inf, -np.inf], np.nan)\n            Xt[c] = Xt[c].astype(float).replace([np.inf, -np.inf], np.nan)\n    return X, Xt, tr[\"target\"].to_numpy().astype(int), tr[\"application_id\"], te[\"application_id\"]\n\n\n# --- model fits: fit(X, y, tr, fit, stop, rest, seed) -> list of predictions for each frame in rest ---------------\n\ndef lgb_fit(params):\n    import lightgbm as lgb\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        p = dict(params, seed=seed)\n        d_fit = lgb.Dataset(X.iloc[tr[fi]], y[tr[fi]])\n        d_st = lgb.Dataset(X.iloc[tr[st]], y[tr[st]], reference=d_fit)\n        m = lgb.train(p, d_fit, N_ROUNDS, valid_sets=[d_st], callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])\n        n = math.ceil(m.best_iteration * REFIT)\n        m = lgb.train(p, lgb.Dataset(X.iloc[tr], y[tr]), n)\n        return [m.predict(r) for r in rest], n\n    return fit\n\n\nLGB_GRID = [  # diverse configurations for nested selection (lgb_nested); chosen per outer fold on inner-stop KS only\n    {}, dict(num_leaves=15, min_child_samples=200), dict(num_leaves=63, min_child_samples=100),\n    dict(num_leaves=127, min_child_samples=300, lambda_l2=20.0), dict(num_leaves=31, feature_fraction=0.3),\n    dict(num_leaves=31, feature_fraction=0.8), dict(num_leaves=7, max_depth=3, min_child_samples=200, feature_fraction=0.4,\n                                                  lambda_l2=10.0), dict(num_leaves=31, min_child_samples=500),\n    dict(num_leaves=255, min_child_samples=500, learning_rate=0.01, lambda_l2=30.0), dict(num_leaves=31, extra_trees=True),\n]\n\n\ndef lgb_nested_fit(grid=LGB_GRID):\n    \"\"\"Clean nested model selection: every config early-stops on the inner stop split; the config with the best\n    inner-stop KS is refitted on the whole training fold (budget x REFIT). No outer-fold label is used.\"\"\"\n    import lightgbm as lgb\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        best = None\n        d_fit = lgb.Dataset(X.iloc[tr[fi]], y[tr[fi]], free_raw_data=False)\n        for j, g in enumerate(grid):\n            p = dict(LGB, **g, seed=seed)\n            m = lgb.train(p, d_fit, N_ROUNDS, valid_sets=[lgb.Dataset(X.iloc[tr[st]], y[tr[st]], reference=d_fit)],\n                          callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])\n            k = ks(y[tr[st]], m.predict(X.iloc[tr[st]], num_iteration=m.best_iteration))\n            if best is None or k > best[0]:\n                best = (k, j, p, m.best_iteration)\n        k, j, p, it = best\n        m = lgb.train(p, lgb.Dataset(X.iloc[tr], y[tr]), math.ceil(it * REFIT))\n        print(f\"lgb_nested: config {j} {grid[j]} inner-stop KS {k:.4f} rounds {it}\", flush=True)\n        return [m.predict(r) for r in rest], j\n    return fit\n\n\ndef xgb_fit(params):\n    import xgboost as xgb\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        p = dict(params, seed=seed)\n        d_fit = xgb.DMatrix(X.iloc[tr[fi]], y[tr[fi]], enable_categorical=True)\n        d_st = xgb.DMatrix(X.iloc[tr[st]], y[tr[st]], enable_categorical=True)\n        m = xgb.train(p, d_fit, N_ROUNDS, evals=[(d_st, \"st\")], early_stopping_rounds=EARLY_STOP, verbose_eval=False)\n        n = math.ceil((m.best_iteration + 1) * REFIT)\n        m = xgb.train(p, xgb.DMatrix(X.iloc[tr], y[tr], enable_categorical=True), n)\n        return [m.predict(xgb.DMatrix(r, enable_categorical=True)) for r in rest], n\n    return fit\n\n\ndef cat_fit(params):\n    from catboost import CatBoostClassifier, Pool\n\n    def s(df):\n        out = df.copy()\n        for c in CAT_COLS:\n            out[c] = out[c].astype(str)\n        return out\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        cats = [c for c in CAT_COLS if c in X.columns]\n        m = CatBoostClassifier(**params, iterations=N_ROUNDS, od_type=\"Iter\", od_wait=EARLY_STOP, use_best_model=True,\n                               random_seed=seed)\n        m.fit(Pool(s(X.iloc[tr[fi]]), y[tr[fi]], cat_features=cats), eval_set=Pool(s(X.iloc[tr[st]]), y[tr[st]], cat_features=cats))\n        n = math.ceil((m.get_best_iteration() + 1) * REFIT)\n        m = CatBoostClassifier(**params, iterations=n, random_seed=seed)\n        m.fit(Pool(s(X.iloc[tr]), y[tr], cat_features=cats))\n        return [m.predict_proba(s(r))[:, 1] for r in rest], n\n    return fit\n\n\ndef tabm_fit(cfg=TABM, device=\"cuda\"):\n    import torch\n    import tabm\n    import rtdl_num_embeddings\n    from sklearn.metrics import roc_auc_score\n    from nn_prep import nn_prep\n\n    def arrays(X, rows_list):\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        return [X.iloc[r][nums].to_numpy(dtype=float) for r in rows_list]\n\n    def codes(df):\n        return np.column_stack([df[c].cat.codes.to_numpy() + 1 for c in CAT_COLS]).astype(np.int64)\n\n    def train(xn, xc, yv, cards, n_epochs, seed, watch=None):\n        torch.manual_seed(seed)\n        dev = torch.device(device)\n        xn_t, xc_t = torch.as_tensor(xn, device=dev), torch.as_tensor(xc, device=dev)\n        yt = torch.as_tensor(yv, dtype=torch.float32, device=dev)\n        emb = rtdl_num_embeddings.PiecewiseLinearEmbeddings(\n            rtdl_num_embeddings.compute_bins(xn_t, n_bins=cfg[\"bins\"]), d_embedding=16, activation=False, version=\"B\")\n        model = tabm.TabM.make(n_num_features=xn.shape[1], cat_cardinalities=cards, d_out=1, num_embeddings=emb).to(dev)\n        opt = torch.optim.AdamW(model.parameters(), lr=cfg[\"lr\"], weight_decay=cfg[\"wd\"])\n\n        def predict(a, c):\n            model.eval()\n            a, c = torch.as_tensor(a, device=dev), torch.as_tensor(c, device=dev)\n            with torch.no_grad():\n                return torch.cat([torch.sigmoid(model(a[i:i + 1024], c[i:i + 1024]).squeeze(-1).float()).mean(1)\n                                  for i in range(0, len(a), 1024)]).cpu().numpy()\n\n        best, best_ep, wait = -1.0, n_epochs, 0\n        for ep in range(1, n_epochs + 1):\n            model.train()\n            for idx in torch.randperm(len(xn_t), device=dev).split(cfg[\"batch\"]):\n                out = model(xn_t[idx], xc_t[idx]).squeeze(-1)\n                loss = torch.nn.functional.binary_cross_entropy_with_logits(out, yt[idx].unsqueeze(1).expand_as(out))\n                opt.zero_grad()\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                opt.step()\n            if watch is not None:\n                auc = roc_auc_score(watch[2], predict(watch[0], watch[1]))\n                best, best_ep, wait = (auc, ep, 0) if auc > best else (best, best_ep, wait + 1)\n                if wait >= cfg[\"patience\"]:\n                    break\n        return model, predict, best_ep\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        cards = [len(X[c].cat.categories) + 1 for c in CAT_COLS]\n        a_fit, a_st = arrays(X, [tr[fi], tr[st]])\n        x_fit, x_st, _ = nn_prep(a_fit, a_st)\n        _, _, ep = train(x_fit, codes(X.iloc[tr[fi]]), y[tr[fi]], cards, cfg[\"epochs\"], seed,\n                         watch=(x_st, codes(X.iloc[tr[st]]), y[tr[st]]))\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        a_tr = X.iloc[tr][nums].to_numpy(dtype=float)\n        a_rest = np.vstack([r[nums].to_numpy(dtype=float) for r in rest])\n        x_tr, x_rest, _ = nn_prep(a_tr, a_rest)\n        model, predict, _ = train(x_tr, codes(X.iloc[tr]), y[tr], cards, ep, seed)\n        p = predict(x_rest, np.vstack([codes(r) for r in rest]))\n        del model\n        gc.collect()\n        torch.cuda.empty_cache()\n        return np.split(p, np.cumsum([len(r) for r in rest])[:-1]), ep\n    return fit\n\n\ndef ks_nan(y, x):\n    x = np.asarray(x, float)\n    if np.isnan(x).all():\n        return 0.0\n    lo, hi = np.nanmin(x) - 1, np.nanmax(x) + 1\n    return max(ks(y, np.where(np.isnan(x), lo, x)), ks(y, np.where(np.isnan(x), hi, x)))\n\n\ndef fm_select(X, y, tr, n_num=100, rand=0, n_top=20):\n    \"\"\"v8 recipe: numeric columns ranked by univariate KS on the training fold, greedy |Spearman| < 0.95, top n_num,\n    plus the 4 categoricals. rand > 0 (feature bagging): the n_top best kept, the other n_num - n_top drawn at random\n    (seed rand) from every column that passes the Spearman filter. rand < 0: rank by LGB gain (seed -rand) instead.\"\"\"\n    nums = [c for c in X.columns if c not in CAT_COLS]\n    sub = X.iloc[tr]\n    if rand < 0:  # rank by LGB split gain on the training fold (conditional usefulness) instead of univariate KS\n        import lightgbm as lgb\n        m = lgb.train(dict(LGB, seed=-rand), lgb.Dataset(sub[nums], y[tr]), 400)\n        gain = dict(zip(nums, m.feature_importance(\"gain\")))\n        ranked = sorted(nums, key=lambda c: -gain[c])[:2 * n_num]\n        rand = 0\n    else:\n        ranked = sorted(nums, key=lambda c: -ks_nan(y[tr], sub[c].to_numpy(dtype=float)))[:None if rand else 2 * n_num]\n    corr = sub[ranked].rank().corr().abs().fillna(0.0)\n    keep = []\n    for c in ranked:\n        if all(corr.loc[c, k] < 0.95 for k in keep):\n            keep.append(c)\n        if len(keep) == n_num and not rand:\n            break\n    if rand:\n        rest = list(np.random.default_rng(rand).permutation(keep[n_top:]))\n        keep = keep[:n_top] + rest[:n_num - n_top]\n    return keep + CAT_COLS\n\n\ndef icl_fit(make, n_num=100, max_context=60000, chunk=10000, as_codes=False, rand=0):\n    \"\"\"In-context learners: no early stopping (the inner split is unused). make(seed, cat_idx) -> sklearn-like model.\"\"\"\n    def fit(X, y, tr, fi, st, rest, seed):\n        cols = fm_select(X, y, tr, n_num, rand)\n        ctx = tr if len(tr) <= max_context else np.random.default_rng(seed).choice(tr, max_context, replace=False)\n\n        def frame(d):\n            d = d[cols].reset_index(drop=True)\n            if as_codes:\n                d = d.copy()\n                for c in CAT_COLS:\n                    d[c] = d[c].cat.codes.astype(float).replace(-1, np.nan)\n                return d.to_numpy(dtype=np.float32)\n            return d\n        m = make(seed, [cols.index(c) for c in CAT_COLS])\n        m.fit(frame(X.iloc[ctx]), y[ctx])\n        preds = [np.concatenate([np.asarray(m.predict_proba(frame(r.iloc[i:i + chunk])))[:, 1]\n                                 for i in range(0, len(r), chunk)]) for r in rest]\n        import torch\n        del m\n        gc.collect()\n        torch.cuda.empty_cache()\n        return preds, len(cols)\n    return fit\n\n\ndef seg_fit(base):\n    \"\"\"Fit `base` separately on thin-file (has_history=0) and history rows: each segment gets its own context and its\n    own fm_select, so thin-file queries are not given ~half all-NaN history columns.\"\"\"\n    def fit(X, y, tr, fi, st, rest, seed):\n        h = X[\"has_history\"].to_numpy()\n        out = [np.zeros(len(r)) for r in rest]\n        for s in (0, 1):\n            ts = tr[h[tr] == s]\n            masks = [r[\"has_history\"].to_numpy() == s for r in rest]\n            preds, n = base(X, y, ts, fi, st, [r[m] for r, m in zip(rest, masks)], seed)\n            for o, p_, m in zip(out, preds, masks):\n                o[m] = p_\n        return out, n\n    return fit\n\n\ndef stack_fit(base, k=5):\n    \"\"\"Stacked ICL: `base` gets one extra column gb_stk = LGB raw score. Context rows (training fold) carry nested k-fold\n    OOF scores from LGB fits inside the training fold; query rows carry the mean score of those same k fits (same\n    training size as the context scores: a refit on the whole fold gave sharper query scores, Causilo fold 0 -0.0027).\n    LGB budget comes from the inner stop split, so no outer-fold label is touched.\"\"\"\n    import lightgbm as lgb\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        p = dict(LGB, seed=seed)\n        m = lgb.train(p, lgb.Dataset(X.iloc[tr[fi]], y[tr[fi]]), N_ROUNDS,\n                      valid_sets=[lgb.Dataset(X.iloc[tr[st]], y[tr[st]])],\n                      callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])\n        n = math.ceil(m.best_iteration * REFIT)\n        g, gq = np.full(len(X), np.nan), [np.zeros(len(r)) for r in rest]\n        for a, b in StratifiedKFold(k, shuffle=True, random_state=seed + 17).split(tr, y[tr]):\n            m = lgb.train(p, lgb.Dataset(X.iloc[tr[a]], y[tr[a]]), n)\n            g[tr[b]] = m.predict(X.iloc[tr[b]], raw_score=True)\n            for q, r in zip(gq, rest):\n                q += m.predict(r, raw_score=True) / k\n        print(f\"stack_fit: gb_stk sd context {np.nanstd(g[tr]):.3f} query {np.std(gq[0]):.3f}\", flush=True)\n        preds, _ = base(X.assign(gb_stk=g), y, tr, fi, st, [r.assign(gb_stk=q) for r, q in zip(rest, gq)], seed)\n        return preds, n\n    return fit\n\n\ndef stack_lr_fit(base, k=5):\n    \"\"\"Supervised oblique direction for trees: column lr_stk = nested logit of lr_q (rank-gauss LR). Training-fold rows\n    carry k-fold OOF lr_q scores fitted inside the training fold; query rows the mean of those k fits. No outer label.\"\"\"\n    lr = lr_q_fit()\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        g, gq = np.full(len(X), np.nan), [np.zeros(len(r)) for r in rest]\n        for a, b in StratifiedKFold(k, shuffle=True, random_state=seed + 23).split(tr, y[tr]):\n            sub = tr[a]\n            f2, s2 = inner(y[sub], seed, 99)\n            preds, _ = lr(X, y, sub, f2, s2, [X.iloc[tr[b]], *rest], seed)\n            g[tr[b]] = preds[0]\n            for q, p_ in zip(gq, preds[1:]):\n                q += p_ / k\n        lg = lambda p_: np.log(np.clip(p_, 1e-6, 1 - 1e-6) / (1 - np.clip(p_, 1e-6, 1 - 1e-6)))  # noqa: E731\n        return base(X.assign(lr_stk=lg(g)), y, tr, fi, st, [r.assign(lr_stk=lg(q)) for r, q in zip(rest, gq)], seed)\n    return fit\n\n\ndef pca_fit(base, k=20):\n    \"\"\"Rotation features for axis-aligned trees: rank-gauss (nn_prep, fitted on the training fold) -> PCA fitted on the\n    training fold -> k components appended as pca_0..k-1. Unsupervised, so no label enters.\"\"\"\n    from nn_prep import nn_prep\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        A_tr, A_all, _ = nn_prep(X.iloc[tr][nums].to_numpy(float), X[nums].to_numpy(float))\n        mu = A_tr.mean(0)\n        _, _, vt = np.linalg.svd(A_tr - mu, full_matrices=False)\n        W = vt[:k].T\n        add = lambda d: d.assign(**{f\"pca_{i}\": c for i, c in enumerate(((nn_prep(X.iloc[tr][nums].to_numpy(float), d[nums].to_numpy(float))[1] - mu) @ W).T)})  # noqa: E731\n        Xa = X.assign(**{f\"pca_{i}\": c for i, c in enumerate(((A_all - mu) @ W).T)})\n        return base(Xa, y, tr, fi, st, [add(r) for r in rest], seed)\n    return fit\n\n\ndef causilo_fit(n_num=100, n_est=4, max_context=60000, device=\"cuda\", rand=0):\n    from causilo import CausiloClassifier\n    return icl_fit(lambda seed, ci: CausiloClassifier(n_estimators=n_est, random_state=seed, device=device),\n                   n_num, max_context, rand=rand)\n\n\ndef tabpfn_fit(n_num=100, n_est=4, max_context=60000, device=\"cuda\"):\n    from tabpfn import TabPFNClassifier\n    return icl_fit(lambda seed, ci: TabPFNClassifier(n_estimators=n_est, random_state=seed, device=device,\n                                                     ignore_pretraining_limits=True, categorical_features_indices=ci,\n                                                     fit_mode=\"fit_with_cache\"),\n                   n_num, max_context, chunk=5000, as_codes=True)\n\n\ndef tabdpt_fit(n_num=100, context=4096, n_ens=8, device=\"cuda\"):\n    \"\"\"TabDPT retrieves `context` nearest training rows per query (its own context reduction).\"\"\"\n    from tabdpt import TabDPTClassifier\n\n    class M:\n        def __init__(self, seed):\n            self.m, self.seed = TabDPTClassifier(device=device, context_reduction=\"retrieval\", missing_indicators=True,\n                                                 compile=False, verbose=False), seed\n\n        def fit(self, Xa, ya):\n            self.m.fit(Xa, ya)\n\n        def predict_proba(self, Xa):\n            return np.mean([self.m.predict_proba(Xa, context_size=context, batch_size=64, seed=self.seed + k) for k in range(n_ens)], axis=0)\n    return icl_fit(lambda seed, ci: M(seed), n_num, 10 ** 9, chunk=2048, as_codes=True)\n\n\ndef kumo_fit(size=\"large\", n_num=100, max_context=20000, n_est=4, chunk=1000, n_bag=1, g_off=0, rand=0):\n    \"\"\"NVIDIA Kumo-Tabular (structured-data-models) in-context learner on fm_select features (categoricals as integer\n    codes, numeric-only model). Context = stratified subsample of `max_context` training-fold rows; no early stopping.\"\"\"\n    import torch\n    import sdm\n\n    def frame(d, cols):\n        d = d[cols].copy()\n        for c in CAT_COLS:\n            d[c] = d[c].cat.codes.astype(float).replace(-1, np.nan)\n        return d.astype(float).reset_index(drop=True)\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        \"\"\"n_bag context draws (stratified, independent) averaged: one 20k draw moves fold KS by up to ~0.01.\"\"\"\n        cols = fm_select(X, y, tr, n_num, rand)\n        dev = torch.device(\"cuda\")\n        model = sdm.models.KumoTabular(task=\"classification\", size=size, device=dev)\n        q = pd.concat([frame(r, cols) for r in rest], ignore_index=True)\n        p = np.mean([one(model, X, y, tr, cols, q, seed * 100 + b, dev, g_off) for b in range(n_bag)], axis=0)\n        del model\n        gc.collect()\n        torch.cuda.empty_cache()\n        return np.split(p, np.cumsum([len(r) for r in rest])[:-1]), len(tr) if len(tr) <= max_context else max_context\n\n    def one(model, X, y, tr, cols, q, seed, dev, g_off=0):\n        rng = np.random.default_rng(seed)\n        ctx = tr if len(tr) <= max_context else np.sort(np.concatenate(\n            [rng.choice(tr[y[tr] == c], int(round(max_context * (y[tr] == c).mean())), replace=False) for c in (0, 1)]))\n        cdf = frame(X.iloc[ctx], cols)\n        full = pd.concat([cdf.assign(target=y[ctx].astype(str)), q.assign(target=\"0\")], ignore_index=True)\n        stypes = sdm.infer_stypes(full, overrides={\"target\": \"categorical\", **{c: \"numerical\" for c in cols}})\n        t = sdm.TableTensor.from_pandas(df=full, stypes=stypes, device=dev)\n        n = len(cdf)\n        xc, yc = t[:n].drop_columns(\"target\"), t[:n, \"target\"]\n        g = torch.Generator(device=dev).manual_seed(seed + 1000 * g_off)  # g_off: independent estimator draw, same context\n        out = []\n        with torch.no_grad(), torch.amp.autocast(\"cuda\", dtype=torch.float16):\n            for i in range(n, len(full), chunk):\n                r = model(x_context=xc, y_context=yc, x_query=t[i:min(i + chunk, len(full))].drop_columns(\"target\"),\n                          num_estimators=n_est, generator=g)\n                out.append(r.to_pandas()[\"1\"].to_numpy())\n        del t\n        return np.concatenate(out)\n    return fit\n\n\ndef _torch_faiss_gpu():\n    \"\"\"shortcut: CPU-only faiss is installed but pytabkit TabR asks for faiss.GpuIndexFlatL2; an exact torch L2 index\n    (cdist + topk on the GPU) stands in. Upgrade to faiss-gpu if TabR becomes part of a recipe.\"\"\"\n    import faiss\n    import torch\n\n    class Index:\n        def __init__(self, *a):\n            self.x = None\n\n        def reset(self):\n            self.x = None\n\n        def add(self, x):\n            self.x = x\n\n        def search(self, q, k):\n            d = torch.cdist(q.float(), self.x.float()) ** 2\n            dist, idx = d.topk(k, dim=1, largest=False)\n            return dist, idx\n\n    faiss.GpuIndexFlatConfig = type(\"Cfg\", (), {\"device\": 0})\n    faiss.StandardGpuResources = lambda: None\n    faiss.GpuIndexFlatL2 = Index\n\n\ndef realmlp_fit(n_ens=1, model=\"RealMLP_TD_Classifier\", **kw):\n    \"\"\"pytabkit RealMLP-TD (tuned defaults; `model` = another pytabkit class, e.g. TabR_S_D_Classifier). Checkpoint\n    selection only on the inner stop split (X_val), then the selected network predicts the outer fold and test (no\n    refit: the schedule is tied to its epoch budget).\"\"\"\n    import pytabkit\n    cls = getattr(pytabkit, model)\n    if model.startswith(\"RealMLP\"):\n        kw |= dict(n_ens=n_ens, val_metric_name=\"1-auc_ovr\")  # TabR only has its default val metric\n    else:\n        _torch_faiss_gpu()\n\n    def prep(X, rows):\n        \"\"\"Median fill fitted on `rows`, one indicator per distinct missingness pattern (thin-file blocks).\"\"\"\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        med = X.iloc[rows][nums].median()\n        miss = X.iloc[rows][nums].isna()\n        groups, seen = [], set()\n        for c in nums:\n            key = np.packbits(miss[c].to_numpy()).tobytes()\n            if miss[c].any() and key not in seen:\n                seen.add(key)\n                groups.append(c)\n\n        def tf(d):\n            out = d[nums].fillna(med).fillna(0.0)\n            for c in groups:\n                out[f\"na_{c}\"] = d[c].isna().astype(float)\n            for c in CAT_COLS:\n                out[c] = d[c]\n            return out\n        return tf\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        tf = prep(X, tr[fi])\n        X = tf(X)\n        rest = [tf(r) for r in rest]\n        cats = CAT_COLS\n        if not model.startswith(\"RealMLP\"):  # pytabkit's cat encoder fails on an empty group under sklearn 1.9\n            X = pd.get_dummies(X, columns=CAT_COLS, dtype=float)\n            rest = [pd.get_dummies(r, columns=CAT_COLS, dtype=float) for r in rest]\n            cats = []\n        m = cls(device=\"cuda\", random_state=seed, verbosity=0, **kw)\n        m.fit(X.iloc[tr[fi]], y[tr[fi]], X_val=X.iloc[tr[st]], y_val=y[tr[st]], cat_col_names=cats)\n        return [m.predict_proba(r)[:, 1] for r in rest], 0\n    return fit\n\n\ndef woe_lr_fit(n_bins=20, alpha=20.0, Cs=(0.003, 0.01, 0.03, 0.1, 0.3)):\n    \"\"\"Credit scorecard: per feature, bins fitted on the training rows (NaN and exact zero get their own bins, else\n    quantile bins of the rest; categoricals by level), smoothed WOE = log odds of the bin shrunk to the prior with\n    `alpha` pseudo-rows, then L2 logistic regression on the standardised WOE columns. C is picked on the inner stop\n    split, then the LR is refitted on the whole training fold (binning refitted there too).\"\"\"\n    from sklearn.linear_model import LogisticRegression\n\n    def encoder(X, rows, yv):\n        prior = yv.mean()\n        lo = np.log(prior / (1 - prior))\n        specs = []\n        for c in X.columns:\n            if c in CAT_COLS:\n                codes = X[c].cat.codes.to_numpy()\n                specs.append((c, \"cat\", None))\n                b = codes[rows]\n            else:\n                x = X[c].to_numpy(dtype=float)[rows]\n                nz = np.isfinite(x) & (x != 0)\n                edges = np.unique(np.quantile(x[nz], np.linspace(0, 1, n_bins + 1))[1:-1]) if nz.sum() > n_bins else np.array([])\n                specs.append((c, \"num\", edges))\n                b = bins_of(x, edges)\n            keys, inv = np.unique(b, return_inverse=True)\n            pos = np.bincount(inv, weights=yv, minlength=len(keys))\n            n = np.bincount(inv, minlength=len(keys))\n            rate = (pos + alpha * prior) / (n + alpha)\n            specs[-1] = specs[-1] + (dict(zip(keys.tolist(), (np.log(rate / (1 - rate)) - lo).tolist())),)\n        return specs\n\n    def bins_of(x, edges):\n        out = np.where(np.isnan(x), -2, np.where(x == 0, -1, np.searchsorted(edges, x, side=\"right\")))\n        return out\n\n    def transform(specs, D):\n        cols = []\n        for c, kind, edges, table in specs:\n            b = D[c].cat.codes.to_numpy() if kind == \"cat\" else bins_of(D[c].to_numpy(dtype=float), edges)\n            cols.append(np.array([table.get(int(k), 0.0) for k in b]) if len(table) < 50 else\n                        np.vectorize(lambda k: table.get(int(k), 0.0))(b))\n        return np.column_stack(cols)\n\n    def fit_lr(A, yv, C, seed):\n        mu, sd = A.mean(0), A.std(0) + 1e-9\n        m = LogisticRegression(C=C, max_iter=2000, random_state=seed).fit((A - mu) / sd, yv)\n        return lambda B: m.decision_function((B - mu) / sd)\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        sp = encoder(X, tr[fi], y[tr[fi]])\n        A, S = transform(sp, X.iloc[tr[fi]]), transform(sp, X.iloc[tr[st]])\n        best = max(Cs, key=lambda C: ks(y[tr[st]], fit_lr(A, y[tr[fi]], C, seed)(S)))\n        sp = encoder(X, tr, y[tr])\n        f = fit_lr(transform(sp, X.iloc[tr]), y[tr], best, seed)\n        return [1 / (1 + np.exp(-f(transform(sp, r)))) for r in rest], best\n    return fit\n\n\ndef lr_q_fit(Cs=(0.001, 0.003, 0.01, 0.03, 0.1, 1.0), onehot=True):\n    \"\"\"Logistic regression on nn_prep inputs (mid-rank ECDF -> normal fitted on the training rows, NaN -> 0, one\n    indicator per missingness pattern) + one-hot categoricals. C picked on the inner stop split, refit on the fold.\"\"\"\n    from sklearn.linear_model import LogisticRegression\n    from nn_prep import nn_prep\n\n    def mats(X, fit_rows, others):\n        nums = [c for c in X.columns if c not in CAT_COLS]\n        a, b, _ = nn_prep(X.iloc[fit_rows][nums].to_numpy(dtype=float),\n                          np.vstack([o[nums].to_numpy(dtype=float) for o in others]))\n        def oh(d):\n            return np.column_stack([(d[c].cat.codes.to_numpy()[:, None] == np.arange(len(d[c].cat.categories))).astype(np.float32)\n                                    for c in CAT_COLS])\n        if onehot:\n            a = np.hstack([a, oh(X.iloc[fit_rows])])\n            b = np.hstack([b, np.vstack([oh(o) for o in others])])\n        return a, np.split(b, np.cumsum([len(o) for o in others])[:-1])\n\n    def fit(X, y, tr, fi, st, rest, seed):\n        a, (s_,) = mats(X, tr[fi], [X.iloc[tr[st]]])\n        best = max(Cs, key=lambda C: ks(y[tr[st]], LogisticRegression(C=C, max_iter=3000).fit(a, y[tr[fi]]).decision_function(s_)))\n        a, outs = mats(X, tr, rest)\n        m = LogisticRegression(C=best, max_iter=3000).fit(a, y[tr])\n        return [m.predict_proba(o)[:, 1] for o in outs], best\n    return fit\n\n\ndef run(name, fit, X, Xt, y, ids, test_ids, out: Path, seeds=SEEDS, folds=None, deadline=None, test=True):\n    \"\"\"Runs fit over seeds x outer folds. OOF = mean over seeds (NaN where no fold ran); test = mean over all fits.\n    Writes out/oof_<name>.parquet, test_<name>.parquet and log_<name>.csv. test=False (screening): no test predictions.\"\"\"\n    out.mkdir(parents=True, exist_ok=True)\n    oof_sum, oof_n, test_sum, log = np.zeros(len(y)), np.zeros(len(y)), np.zeros(len(Xt)), []\n    for seed in seeds:\n        for k, (tr, va) in enumerate(outer(y, seed)):\n            if folds is not None and k not in folds:\n                continue\n            if deadline is not None and time.time() > deadline:\n                raise TimeoutError(f\"{name}: deadline before seed {seed} fold {k}\")\n            t0 = time.time()\n            fi, st = inner(y[tr], seed, k)\n            preds, budget = fit(X, y, tr, fi, st, [X.iloc[va], Xt] if test else [X.iloc[va]], seed)\n            pv, pt = preds[0], preds[1] if test else np.nan\n            oof_sum[va] += pv\n            oof_n[va] += 1\n            test_sum += pt\n            log.append(dict(model=name, seed=seed, fold=k, ks=ks(y[va], pv), budget=budget, sec=round(time.time() - t0, 1)))\n            print(log[-1], flush=True)\n            pd.DataFrame(log).to_csv(out / f\"log_{name}.csv\", index=False)\n            # written after every fold so an interrupted run keeps the folds it finished\n            oof = np.where(oof_n > 0, oof_sum / np.maximum(oof_n, 1), np.nan)\n            pl.DataFrame({\"application_id\": ids, \"target\": y, name: oof}).write_parquet(out / f\"oof_{name}.parquet\")\n            if test:\n                pl.DataFrame({\"application_id\": test_ids, name: test_sum / len(log)}).write_parquet(out / f\"test_{name}.parquet\")\n    m = oof_n > 0\n    print(f\"{name}: OOF KS {ks(y[m], oof[m]):.5f} on {m.sum()} rows\", flush=True)\n    return oof, test_sum / len(log), log\n\n\ndef manifest(out: Path, extra=None):\n    import importlib.metadata as md\n    pk = {}\n    for n in [\"numpy\", \"pandas\", \"polars\", \"scikit-learn\", \"scipy\", \"lightgbm\", \"xgboost\", \"catboost\", \"torch\", \"tabm\",\n              \"rtdl_num_embeddings\", \"causilo\"]:\n        try:\n            pk[n] = md.version(n)\n        except md.PackageNotFoundError:\n            pk[n] = None\n    cfg = dict(N_FOLDS=N_FOLDS, INNER=INNER, REFIT=REFIT, SEEDS=SEEDS, N_ROUNDS=N_ROUNDS, EARLY_STOP=EARLY_STOP,\n               LGB=LGB, LGB_S=LGB_S, XGB=XGB, XGB_S=XGB_S, CAT=CAT, CAT_S=CAT_S, TABM=TABM)\n    (out / \"manifest.json\").write_text(json.dumps(dict(packages=pk, config=cfg, **(extra or {})), indent=2, default=str))\n", "251_repayment_markov.py": "\"\"\"Bet 2: per-customer repayment Markov chain and analytic P(reach 90+ DPD within 12 installments).\n\nStates per installment from max_days_past_due: 0 (on time), 1-29, 30-59, 60-89, 90+. Transitions are counted between\nconsecutive installments of the same historical loan (all rows are before the selected decision by construction).\nEach customer's transition rows are shrunk to the population matrix with a Dirichlet prior (no label used).\nThe 90+ state is made absorbing to get the probability that a fresh 12-installment loan ever reaches it.\n\nReports KS and KS within deciles of the strongest existing repayment features.\n\"\"\"\nimport numpy as np\nimport polars as pl\n\nimport common as c\n\nEDGES = [0, 29, 59, 89]\nK = 5\nALPHA = 5.0\nSTEPS = 12\n\n\ndef hit_probability(P, start=0, steps=STEPS):\n    \"\"\"P: (n, K, K) row-stochastic. Probability of being absorbed in state K-1 within `steps` from `start`.\"\"\"\n    A = P.copy()\n    A[:, K - 1, :] = 0.0\n    A[:, K - 1, K - 1] = 1.0\n    v = np.zeros((len(P), K))\n    v[:, start] = 1.0\n    out = []\n    for _ in range(steps):\n        v = np.einsum(\"nk,nkj->nj\", v, A)\n        out.append(v[:, K - 1].copy())\n    return np.stack(out, 1)\n\n\ndef main():\n    f = c.features()\n    apps = f.select(\"application_id\", \"customer_id\", \"target\", \"is_train\")\n    r = (c.load(\"repayment_history\").sort(\"customer_id\", \"loan_id\", \"due_date\")\n         .with_columns(s=pl.sum_horizontal([(pl.col(\"max_days_past_due\") > e).cast(pl.Int8) for e in EDGES]))\n         .with_columns(prev=pl.col(\"s\").shift(1).over(\"loan_id\")))\n    tr = r.drop_nulls(\"prev\").group_by(\"customer_id\", \"prev\", \"s\").len()\n\n    cust = apps[\"customer_id\"].to_list()\n    pos = {cid: i for i, cid in enumerate(cust)}\n    N = np.zeros((len(cust), K, K))\n    ci = np.array([pos[x] for x in tr[\"customer_id\"].to_list()])\n    N[ci, tr[\"prev\"].to_numpy(), tr[\"s\"].to_numpy()] = tr[\"len\"].to_numpy()\n\n    pop = N.sum(0)\n    pop = pop / pop.sum(1, keepdims=True)\n    post = (ALPHA * pop[None] + N) / (ALPHA + N.sum(2, keepdims=True))\n    has = N.sum((1, 2)) > 0\n\n    hit = hit_probability(post)\n    n_from0 = N[:, 0].sum(1)\n    n_late = N[:, 1:].sum((1, 2))\n    feats = {\n        \"mk_hit12\": hit[:, -1],\n        \"mk_hit6\": hit[:, 5],\n        \"mk_hit3\": hit[:, 2],\n        \"mk_entry\": 1 - post[:, 0, 0],\n        \"mk_cure\": post[:, 1:4, 0].mean(1),\n        \"mk_escalate\": (post[:, 1, 2:].sum(1) + post[:, 2, 3:].sum(1) + post[:, 3, 4]) / 3,\n        \"mk_stay90\": post[:, 4, 4],\n        \"mk_n_trans\": N.sum((1, 2)),\n        \"mk_n_from0\": n_from0,\n        \"mk_n_late\": n_late,\n    }\n    for k in feats:\n        if k.startswith(\"mk_\") and not k.startswith(\"mk_n\"):\n            feats[k] = np.where(has, feats[k], np.nan)\n    out = apps.select(\"application_id\").with_columns(**{k: pl.Series(v) for k, v in feats.items()})\n    out.write_parquet(c.PQ / \"f251_repayment_markov.parquet\")\n\n    print(\"population transition matrix (rows = from state 0,1-29,30-59,60-89,90+):\")\n    print(np.round(pop, 3))\n    print(f\"customers with transitions: {has.mean():.3f}\")\n\n    d = (f.filter(pl.col(\"is_train\") == 1)\n         .select(\"application_id\", \"target\", \"rp_frac_dpd1\", \"rp_maxdpd_mean\", \"rq_streak_max\", \"sv_share30_of_late\")\n         .join(out, on=\"application_id\"))\n    y = d[\"target\"].to_numpy()\n    hist = ~np.isnan(d[\"mk_hit12\"].to_numpy())\n    print(f\"train rows with history {hist.sum()} of {len(d)}; bad rate with/without history \"\n          f\"{y[hist].mean():.4f}/{y[~hist].mean():.4f}\")\n\n    def deciles(col):\n        x = d[col].to_numpy().astype(float)[hist]\n        qs = np.unique(np.nanquantile(x, np.linspace(.1, .9, 9)))\n        return np.digitize(x, qs)\n\n    dec_a, dec_b = deciles(\"rp_frac_dpd1\"), deciles(\"rp_maxdpd_mean\")\n    yh = y[hist]\n    print(f\"{'feature (history only)':24s} {'KS':>7s} {'|frac_dpd1':>11s} {'|maxdpd_mean':>13s}\")\n    for col in [\"rp_frac_dpd1\", \"rp_maxdpd_mean\", \"rq_streak_max\", \"sv_share30_of_late\"] + list(feats):\n        s = d[col].to_numpy().astype(float)[hist]\n        within = []\n        for dec in (dec_a, dec_b):\n            ks = [c.ks(yh[dec == i], s[dec == i]) for i in np.unique(dec)\n                  if (dec == i).sum() > 300 and 0 < yh[dec == i].mean() < 1 and np.isfinite(s[dec == i]).any()]\n            within.append(np.mean(ks))\n        print(f\"{col:24s} {c.ks(yh, s):7.4f} {within[0]:11.4f} {within[1]:13.4f}\")\n\n\nif __name__ == \"__main__\":\n    main()\n", "kaggle_v72_run.py": "\"\"\"v72 Kaggle run: 9 GBDT (6 v22 family + lgb_x, xgb_d8, cat_d8) on main features + f251 Markov, seeds 42/7/2026.\nOne process per device group so LightGBM (CPU), XGBoost (GPU 0) and CatBoost (GPU 1) run side by side.\nUsage: python kaggle_v72_run.py lgb|xgb|cat [seed,seed,...] [model,model,...]\n(features.parquet and f251 must already exist in data_pq/; defaults: valid2 seeds, all models in the group)\n\"\"\"\nimport sys\n\nimport polars as pl\n\nimport valid2 as v\nfrom common import PQ, ROOT\n\nOUT = ROOT / \"v72\"\nGROUPS = {\n    \"lgb\": {\"lgb\": lambda: v.lgb_fit(dict(v.LGB, n_jobs=2)), \"lgb_s\": lambda: v.lgb_fit(dict(v.LGB_S, n_jobs=2)),\n            \"lgb_x\": lambda: v.lgb_fit(dict(v.LGB_X, n_jobs=2))},\n    \"xgb\": {\"xgb\": lambda: v.xgb_fit(v.XGB), \"xgb_s\": lambda: v.xgb_fit(v.XGB_S), \"xgb_d8\": lambda: v.xgb_fit(v.XGB_D8)},\n    \"cat\": {\"cat\": lambda: v.cat_fit(v.CAT), \"cat_s\": lambda: v.cat_fit(v.CAT_S), \"cat_d8\": lambda: v.cat_fit(v.CAT_D8)},\n}\n\n\ndef main():\n    feats = pl.read_parquet(PQ / \"features.parquet\").join(\n        pl.read_parquet(PQ / \"f251_repayment_markov.parquet\"), on=\"application_id\", how=\"left\")\n    X, Xt, y, ids, test_ids = v.frames(feats)\n    print(f\"{X.shape[1]} features, {len(y)} train rows, {len(Xt)} test rows\", flush=True)\n    seeds = [int(s) for s in sys.argv[2].split(\",\")] if len(sys.argv) > 2 else v.SEEDS\n    keep = sys.argv[3].split(\",\") if len(sys.argv) > 3 else None\n    for name, make in GROUPS[sys.argv[1]].items():\n        if keep is None or name in keep:\n            v.run(name, make(), X, Xt, y, ids, test_ids, OUT, seeds=seeds)\n\n\nif __name__ == \"__main__\":\n    main()\n"}
for name, text in FILES.items():
    (CODE / name).write_text(text)
print(sorted(os.listdir(CODE)))

In [ ]:
import subprocess, sys
def sh(args, log=None):
    r = subprocess.run(args, cwd=CODE, capture_output=True, text=True)
    print(r.stdout[-4000:], r.stderr[-4000:])
    assert r.returncode == 0
sh([sys.executable, '-c', 'import common as c; f = c.features(); print(f.shape)'])
sh([sys.executable, '251_repayment_markov.py'])

In [ ]:
import polars as pl
f = pl.read_parquet('/kaggle/working/code/data_pq/features.parquet').sort('application_id')
num = [c for c, d in f.schema.items() if d.is_numeric()]
sums = f.select([pl.col(c).cast(pl.Float64).fill_nan(None).sum().alias(c) for c in num])
os.makedirs('/kaggle/working/v72', exist_ok=True)
sums.write_parquet('/kaggle/working/v72/feature_sums.parquet')
print(f.shape)

In [ ]:
import time
env = dict(os.environ)
procs = {}
for g, gpu in (('lgb', ''), ('xgb', '0'), ('cat', '1')):
    e = dict(env, CUDA_VISIBLE_DEVICES=gpu)
    procs[g] = subprocess.Popen([sys.executable, 'kaggle_v72_run.py', g] + ['1,11,101,777,31337', 'lgb,lgb_s,xgb,xgb_s,cat,cat_s'], cwd=CODE, env=e,
                                stdout=open(f'/kaggle/working/v72/run_{g}.log', 'w'), stderr=subprocess.STDOUT)
t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(60)
    print(f'{(time.time() - t0) / 60:.0f} min', {g: p.poll() for g, p in procs.items()}, flush=True)
for g in procs:
    print(g, procs[g].returncode)
    print(open(f'/kaggle/working/v72/run_{g}.log').read()[-1500:])

In [ ]:
import numpy as np
from scipy.stats import rankdata, ks_2samp
OUT = pathlib.Path('/kaggle/working/code/v72')
ss = pl.read_csv(next(pathlib.Path('/kaggle/input').rglob('sample_submission.csv')))
def blend(models, fname):
    oof = [pl.read_parquet(OUT / f'oof_{m}.parquet') for m in models]
    y = oof[0]['target'].to_numpy()
    o = np.mean([rankdata(d[m].to_numpy()) for d, m in zip(oof, models)], 0)
    ks = ks_2samp(o[y == 1], o[y == 0]).statistic
    t = ss.select('application_id')
    for m in models:
        t = t.join(pl.read_parquet(OUT / f'test_{m}.parquet'), on='application_id', how='left')
    r = np.mean([rankdata(t[m].to_numpy()) / t.height for m in models], 0)
    t.select('application_id').with_columns(default_probability=pl.Series(r)).write_csv(f'/kaggle/working/{fname}')
    print(fname, 'OOF KS', round(ks, 5), [round(ks_2samp(d[m].to_numpy()[y == 1], d[m].to_numpy()[y == 0]).statistic, 5) for d, m in zip(oof, models)])
six = ['lgb', 'lgb_s', 'xgb', 'xgb_s', 'cat', 'cat_s']
blend(six, 'submission_six.csv')
import shutil
shutil.copytree(OUT, '/kaggle/working/v72/preds', dirs_exist_ok=True)